In [ ]:
import pandas as pd
import duckdb

# 投料数据：电商秒杀节注册与首单流
user_orders = pd.DataFrame({
    'order_id': ['o_01', 'o_02', 'o_03', 'o_04'],
    'user_id': [9001, 9002, 9003, 9004],
    'signup_time': ['2026-06-21 00:00:00', '2026-06-21 06:00:00', '2026-06-21 12:00:00', '2026-06-21 23:00:00'],
    'order_time': ['2026-06-21 01:15:00', '2026-06-21 10:30:00', '2026-06-21 12:05:00', '2026-06-22 03:00:00']
})

### 🎯 2. 核心刚性需求

1. **锁定风控安全线**：利用 `INTERVAL` 算子，计算出每个用户注册后的**黑产高危观察截止时间**（即 `signup_time` 往后平移 **2 小时**）。
    
2. **大闸拦截**：如果下单时间（`order_time`）严格落在了注册后的 2 小时观察期内，将其判定为高危单。
    
3. **提取特征**：利用 `EXTRACT(EPOCH FROM ...)` 或者是 Pandas 的平替算子，算出这批高危单到底是在注册后**多少秒内**发起的，命名为 `fraud_window_seconds`。
    
4. **最终输出**：输出被逮住的 `user_id` 和 `fraud_window_seconds`。